# Python Idioms Tour (12 min)

Live-coded patterns you will reuse all day:

- with blocks and context managers
- pathlib path arithmetic and file IO
- list and dict comprehensions
- *args and **kwargs
- enumerate and zip
- Pydantic with one ClaimsRecord model

In [ ]:
# Python imports are module-first and explicit.
# JS example: import fs from 'node:fs';  Java example: import java.nio.file.Path;
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import date
import csv

project_root = Path.cwd()
claims_csv = project_root / "data" / "claims_sample.csv"

print("Project root:", project_root)
print("Claims file exists:", claims_csv.exists())

## 1) with blocks and context managers

Use `with` to guarantee cleanup, even on errors.

In [ ]:
# Python 'with' auto-closes resources via context managers.
# JS usually: const f = await fs.open(...); try { ... } finally { await f.close(); }
# Java usually: try (var br = Files.newBufferedReader(path)) { ... }
with claims_csv.open("r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    first_two = [next(reader) for _ in range(2)]

print("Read rows with automatic file close:")
for row in first_two:
    print(row["policy_id"], row["claim_amount"], row["region"])

## 2) pathlib for path arithmetic and file IO

`Path` objects make joins, reads, and writes clear and safe.

In [ ]:
# pathlib uses '/' for path composition and stays cross-platform.
# JS often uses path.join('data', 'claims_sample.csv');
# Java/C# often use Paths.get(...) or Path.Combine(...).
data_dir = project_root / "data"
input_file = data_dir / "claims_sample.csv"

print("Input file name:", input_file.name)
print("Input suffix:", input_file.suffix)

with TemporaryDirectory() as tmp:
    out_file = Path(tmp) / "preview.txt"
    out_file.write_text("pathlib writes this file\n", encoding="utf-8")
    print("Temp output exists inside context:", out_file.exists())

print("Temp output cleaned automatically after context exit.")

## 3) list and dict comprehensions

Compact transformations and filtered mappings.

In [ ]:
amounts = [280032.31, 22195.18, 9624.17, 8865.12, 500.0]
perils = ["theft", "theft", "water_damage", "fire", "fire"]

# Python comprehensions are expression-based and concise.
# JS equivalent: amounts.filter(a => a >= 10000)
# Java equivalent: amounts.stream().filter(a -> a >= 10000).toList()
large_claims = [a for a in amounts if a >= 10000]
peril_counts = {p: perils.count(p) for p in sorted(set(perils))}

print("Large claims:", large_claims)
print("Peril counts:", peril_counts)

## 4) `*args` and `**kwargs`

Use flexible signatures to pass variable positional and named data.

In [ ]:
# *args packs extra positional args into a tuple.
# **kwargs packs extra named args into a dict.
# JS has rest params (...args), but no direct **kwargs equivalent.
def summarize_claims(*claim_amounts: float, currency: str = "EUR", **meta):
    total = sum(claim_amounts)
    average = total / len(claim_amounts) if claim_amounts else 0.0
    return {
        "count": len(claim_amounts),
        "total": round(total, 2),
        "average": round(average, 2),
        "currency": currency,
        "meta": meta,
    }

summary = summarize_claims(280032.31, 22195.18, 9624.17, currency="EUR", source="workshop", batch=1)
print(summary)

## 5) `enumerate` and `zip`

`enumerate` gives position, `zip` aligns sequences.

In [ ]:
regions = ["DE-BAY", "NL-NH", "DE-NRW", "NL-ZH"]
recent_perils = ["theft", "theft", "water_damage", "fire"]

# enumerate adds an index without manual counters.
# zip pairs lists by position (like JS map with index + second array).
for idx, (region, peril) in enumerate(zip(regions, recent_perils), start=1):
    print(f"{idx:02d}. {region} -> {peril}")

## 6) Pydantic: typed struct + runtime validation

This model becomes a reusable shape for config and records later today and on Day 2.

In [ ]:
# Pydantic is a typed model with runtime validation/parsing.
# Unlike plain dataclasses (no validation by default) or TS interfaces (compile-time only).
from pydantic import BaseModel, ValidationError

class ClaimsRecord(BaseModel):
    policy_id: str
    claim_date: date
    peril: str
    region: str
    claim_amount: float
    premium: float
    loss_year: int

with claims_csv.open("r", encoding="utf-8") as f:
    first_row = next(csv.DictReader(f))

record = ClaimsRecord.model_validate(first_row)
print("Validated ClaimsRecord:", record)

try:
    ClaimsRecord.model_validate({**first_row, "claim_amount": "not_a_number"})
except ValidationError as e:
    print("Validation error example:")
    print(e)